[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/experimental-psychology/blob/main/notebooks/picture_lab_analysis.ipynb)

# Picture Lab: Instruction Effectiveness Analysis

This notebook provides tools for analyzing data from the Picture Lab. You'll evaluate how effectively groups' drawing instructions were followed and what factors predict instruction quality.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set plot style for clean figures
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## Loading Evaluation Data

We'll load data from the class Google Sheet. To get a direct CSV link from a Google Sheet:

1. Open the Google Sheet
2. Go to **File > Share > Publish to web**
3. Choose the relevant sheet/tab and select **CSV** as the format
4. Copy the published URL and paste it below

Alternatively, you can download the sheet as a CSV and upload it to Colab using the file browser on the left sidebar.

In [ ]:
# === Step-Following Data ===
# Rows = groups whose instructions were being followed
# Columns = individual steps in those instructions
# Values = 1 if the step was followed correctly, 0 if not

# Option 1: Load from a published Google Sheet URL
# step_data = pd.read_csv('YOUR_GOOGLE_SHEET_CSV_URL_HERE')

# Option 2: Load from an uploaded CSV file
# step_data = pd.read_csv('step_following_data.csv', index_col=0)

# --- Example data for testing (replace with real data) ---
np.random.seed(42)
group_names = [f'Group {i}' for i in range(1, 7)]
step_labels = [f'Step {i}' for i in range(1, 11)]

# Simulate binary step-following outcomes
step_data = pd.DataFrame(
    np.random.choice([0, 1], size=(len(group_names), len(step_labels)), p=[0.3, 0.7]),
    index=group_names,
    columns=step_labels
)

print('Step-following data (1 = followed correctly, 0 = not):')
step_data

In [ ]:
# Calculate per-group accuracy: proportion of steps followed correctly
accuracy = step_data.mean(axis=1)

accuracy_df = pd.DataFrame({
    'Group': accuracy.index,
    'Accuracy': accuracy.values,
    'Steps Correct': step_data.sum(axis=1).values,
    'Total Steps': step_data.shape[1]
})

print('Per-group instruction accuracy:')
print(accuracy_df.to_string(index=False))
print(f'\nOverall mean accuracy: {accuracy.mean():.2%}')
print(f'Standard deviation: {accuracy.std():.2%}')

In [ ]:
# Heatmap: which steps were followed/not-followed by each group
fig, ax = plt.subplots(figsize=(12, 5))

sns.heatmap(
    step_data,
    annot=True,
    fmt='d',
    cmap='RdYlGn',
    cbar_kws={'label': 'Followed Correctly'},
    linewidths=0.5,
    ax=ax
)

ax.set_title('Step-Following Results by Group', fontsize=14)
ax.set_xlabel('Instruction Step')
ax.set_ylabel('Group (whose instructions were followed)')
plt.tight_layout()
plt.show()

In [ ]:
# === Ratings Data ===
# Each group's instructions are rated on four dimensions (1-10 scale):
#   - Appearance: how closely the drawing matched the original picture
#   - Meaning: how well the drawing conveyed the intended meaning
#   - Clarity: how clear/unambiguous the instructions were
#   - Efficiency: how concise the instructions were (fewer steps for same result)

# Option 1: Load from a published Google Sheet URL
# ratings = pd.read_csv('YOUR_GOOGLE_SHEET_CSV_URL_HERE')

# Option 2: Load from an uploaded CSV file
# ratings = pd.read_csv('ratings_data.csv', index_col=0)

# --- Example data for testing (replace with real data) ---
np.random.seed(99)
ratings = pd.DataFrame({
    'Group': group_names,
    'Appearance': np.random.randint(3, 10, size=len(group_names)),
    'Meaning': np.random.randint(3, 10, size=len(group_names)),
    'Clarity': np.random.randint(2, 10, size=len(group_names)),
    'Efficiency': np.random.randint(2, 10, size=len(group_names)),
    'Num_Assumptions': np.random.randint(0, 8, size=len(group_names)),
    'Num_Instructions': np.random.randint(5, 20, size=len(group_names))
}).set_index('Group')

print('Instruction ratings (1-10 scale):')
ratings

In [ ]:
# Bar chart: comparing groups on clarity vs efficiency ratings
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(ratings.index))
width = 0.35

bars1 = ax.bar(x - width/2, ratings['Clarity'], width, label='Clarity', color='steelblue')
bars2 = ax.bar(x + width/2, ratings['Efficiency'], width, label='Efficiency', color='coral')

ax.set_xlabel('Group')
ax.set_ylabel('Rating (1-10)')
ax.set_title('Clarity vs. Efficiency Ratings by Group', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(ratings.index, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 11)

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: number of assumptions vs. clarity rating
# Question: do more assumptions correlate with less clear instructions?

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(ratings['Num_Assumptions'], ratings['Clarity'], s=100, color='steelblue', edgecolors='black')

# Label each point with the group name
for group in ratings.index:
    ax.annotate(group,
                (ratings.loc[group, 'Num_Assumptions'], ratings.loc[group, 'Clarity']),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

# Fit and plot a regression line
slope, intercept, r_value, p_value, std_err = stats.linregress(
    ratings['Num_Assumptions'], ratings['Clarity']
)
x_line = np.linspace(ratings['Num_Assumptions'].min() - 0.5, ratings['Num_Assumptions'].max() + 0.5, 100)
ax.plot(x_line, intercept + slope * x_line, '--', color='gray', alpha=0.7)

ax.set_xlabel('Number of Assumptions Made', fontsize=12)
ax.set_ylabel('Clarity Rating (1-10)', fontsize=12)
ax.set_title('Do More Assumptions Lead to Less Clear Instructions?', fontsize=14)
ax.text(0.05, 0.95, f'r = {r_value:.2f}, p = {p_value:.3f}',
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Correlation: number of instructions vs. accuracy
# Question: do more detailed instructions help or hurt?

# Merge accuracy with ratings data
combined = ratings.copy()
combined['Accuracy'] = accuracy

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(combined['Num_Instructions'], combined['Accuracy'], s=100, color='coral', edgecolors='black')

# Label each point
for group in combined.index:
    ax.annotate(group,
                (combined.loc[group, 'Num_Instructions'], combined.loc[group, 'Accuracy']),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

# Fit and plot a regression line
slope, intercept, r_value, p_value, std_err = stats.linregress(
    combined['Num_Instructions'], combined['Accuracy']
)
x_line = np.linspace(combined['Num_Instructions'].min() - 1, combined['Num_Instructions'].max() + 1, 100)
ax.plot(x_line, intercept + slope * x_line, '--', color='gray', alpha=0.7)

ax.set_xlabel('Number of Instructions', fontsize=12)
ax.set_ylabel('Accuracy (proportion of steps followed)', fontsize=12)
ax.set_title('Do More Detailed Instructions Improve Accuracy?', fontsize=14)
ax.text(0.05, 0.95, f'r = {r_value:.2f}, p = {p_value:.3f}',
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

# Print summary
print(f'Correlation between number of instructions and accuracy:')
print(f'  r = {r_value:.3f}, p = {p_value:.3f}')
if p_value < 0.05:
    direction = 'positively' if r_value > 0 else 'negatively'
    print(f'  => Statistically significant: more instructions are {direction} associated with accuracy.')
else:
    print(f'  => Not statistically significant at p < 0.05.')

## Discussion: From Drawing Instructions to Methods Sections

The Picture Lab highlights a core challenge in science: **writing instructions that someone else can follow to reproduce your results**. This is exactly what a Methods section in a scientific paper must do.

Consider the following parallels:

| Picture Lab | Scientific Methods Section |
|-|-|
| Drawing instructions | Experimental protocol |
| Assumptions about shared knowledge | Assumed expertise of the reader |
| Step-by-step clarity | Procedural detail and reproducibility |
| Efficiency (fewer steps) | Conciseness and readability |
| Accuracy of the final drawing | Reproducibility of results |

### Reflection Questions

1. **Clarity vs. efficiency trade-off:** Did groups with more concise instructions sacrifice clarity? Is there an optimal balance?

2. **Assumptions:** Which assumptions were most problematic? How does this relate to writing for an audience with different levels of expertise?

3. **Specificity:** Did groups with more steps actually achieve better accuracy, or did excessive detail cause confusion?

### Try This: Use GenAI to Compare Approaches

Copy your group's drawing instructions into a GenAI tool (e.g., ChatGPT, Claude) and ask it to:

- Identify ambiguous steps that could be interpreted multiple ways
- Suggest how to rewrite the instructions for maximum clarity
- Compare your instructions to another group's and explain which set would be easier to follow and why

Then try the same exercise with a real Methods section from a published paper in your area of interest. How do the same principles of clarity, assumptions, and efficiency apply?